In [9]:
import pandas as pd

df1 = pd.read_excel('data/format_split.xlsx', sheet_name='fmt115')
df2 = pd.read_excel('data/資料清洗模組(20260414V1).xlsx', sheet_name='附件二欄位名稱對照表')

df1['FieldID'] = df1['FieldID'].astype(str).str.strip()
df2['序號'] = df2['序號'].astype(str).str.strip()


merged = pd.merge(df1,df2,left_on='FieldID',right_on='序號',how='left')
result = merged[['FieldID','ChineseName','EnglishName',]]

output_file = '合併結果.xlsx'

with pd.ExcelWriter('MergeResult.xlsx', engine='openpyxl') as writer:
    result.to_excel(writer, sheet_name='Merge_Result', index=False)

print(result.head(10))

  FieldID ChineseName              EnglishName
0     1.1      申報醫院代碼  Reporting Hospital Code
1     1.2        病歷號碼    Medical Record Number
2     1.3          姓名                     Name
3     1.4     身分證統一編號                ID Number
4     1.5          性別                      Sex
5     1.6        出生日期            Date of Birth
6     1.7       戶籍地代碼           Residence Code
7     2.1        診斷年齡         Age at Diagnosis
8     2.2    癌症發生順序號碼          Sequence Number
9     2.3        個案分類            Class of Case


In [2]:
#file_name = "test/DeclarationFormat42.y_2019129.txt"
#file_name = "test/DeclarationFormat45.20102024y_20251124.txt"
#file_name = "test/DeclarationFormat50.20242024y_2026126.txt"
file_name = "test/DeclarationFormat114.20242024y_2026417.txt"
#file_name = "test/DeclarationFormat115.20102024y_20251211.txt"
#file_name = "test/DeclarationFormat129.20252025y_2026320.txt"


with open(file_name, "r", encoding="big5") as f:
    for i, line in enumerate(f):
        clean_line = line.strip('\n').strip('\r')
        length = len(clean_line.encode('big5'))
        print(f"第 {i+1} 行長度: {length}")

第 1 行長度: 402
第 2 行長度: 402
第 3 行長度: 402
第 4 行長度: 402
第 5 行長度: 402
第 6 行長度: 402
第 7 行長度: 402
第 8 行長度: 402
第 9 行長度: 402
第 10 行長度: 402
第 11 行長度: 402
第 12 行長度: 402
第 13 行長度: 402
第 14 行長度: 402
第 15 行長度: 402
第 16 行長度: 402
第 17 行長度: 402
第 18 行長度: 402
第 19 行長度: 402
第 20 行長度: 402
第 21 行長度: 402
第 22 行長度: 402
第 23 行長度: 402
第 24 行長度: 402
第 25 行長度: 402
第 26 行長度: 402
第 27 行長度: 402
第 28 行長度: 402
第 29 行長度: 402
第 30 行長度: 402


In [4]:
import csv
from modules.db import get_conn

def load_field_spec(fmt):
    conn = get_conn()
    cursor = conn.cursor()
    sql = """SELECT [ChineseName],[Start],[End] FROM CancerRegistry_Fields WHERE [fmt]=? ORDER BY [Start]"""
    cursor.execute(sql, fmt)
    rows = cursor.fetchall()
    conn.close()
    field_spec = [(r[0], int(r[1]), int(r[2])) for r in rows]
    return field_spec


def parse_fixed_width_line(line_text, spec):
    line_bytes = line_text.encode('big5', errors='ignore')
    parsed_row = {}
    for name, start, end in spec:
        field_value = line_bytes[start - 1:end]
        try:
            parsed_row[name] = field_value.decode('big5').strip()
        except:
            parsed_row[name] = field_value.decode('big5', errors='replace').strip()
    return parsed_row


def main(input_file, output_csv, fmt):
    results = []

    field_spec = load_field_spec(fmt)
    with open(input_file, 'r', encoding='big5') as f:
        for i, line in enumerate(f, 1):
            clean_line = line.replace('\n', '').replace('\r', '')
            if not clean_line.strip():
                continue

            parsed_data = parse_fixed_width_line(clean_line, field_spec)
            results.append(parsed_data)
            if i % 100 == 0:
                print(f"已處理 {i} 行...")
    print(f"總共 {len(results)} 筆資料")


    if results:
        keys = [f[0] for f in field_spec]
        with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(results)
        print(f"CSV 已輸出: {output_csv}")


if __name__ == "__main__":
    file_name = "test/DeclarationFormat114.20242024y_2026417.txt"
    fmt=114
    output_filename = f"申報資料解析結果{fmt}.csv"
    main(file_name, output_filename, fmt)

總共 30 筆資料
CSV 已輸出: 申報資料解析結果114.csv


In [30]:
conn = get_conn()
cursor = conn.cursor()
sql = """SELECT MAX([End]) FROM [Hospital_data].[dbo].[CancerRegistry_Fields] WHERE fmt=? GROUP BY [fmt]"""
cursor.execute(sql, 42)
rows = cursor.fetchone()

print(rows[0])
conn.close()

198


In [1]:
import csv
from modules.db import get_conn

def load_field_spec(fmt=42):
    conn = get_conn()
    cursor = conn.cursor()
    sql = """SELECT [ChineseName],[Start],[End] FROM CancerRegistry_Fields WHERE [fmt]=? ORDER BY [Start]"""
    cursor.execute(sql, fmt)
    rows = cursor.fetchall()
    conn.close()
    field_spec = [(r[0],int(r[1]),int(r[2])) for r in rows]
    return field_spec

def parse_fixed_width_line(line_text, spec):
    line_bytes = line_text.encode('big5', errors='ignore')
    parsed_row = {}
    for name, start, end in spec:
        field_value = line_bytes[start - 1:end]
        try:
            parsed_row[name] = field_value.decode('big5').strip()
        except:
            parsed_row[name] = field_value.decode('big5', errors='replace').strip()
    return parsed_row


def main(input_file, output_csv, fmt):
    results = []

    field_spec = load_field_spec(fmt)
    with open(input_file, 'r', encoding='big5') as f:
        for i, line in enumerate(f, 1):
            # clean_line = line.replace('\n', '').replace('\r', '')
            # if not clean_line.strip():
            #     continue

            parsed_data = parse_fixed_width_line(line, field_spec)
            results.append(parsed_data)
    print(f"count:{len(results)}row data")

    if results:
        keys = [f[0] for f in field_spec]
        with open(output_csv, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(results)
        # print(f"csv: {output_csv}")


if __name__ == "__main__":
    file_name = "test/DeclarationFormat115.20102024y_20251211.txt"
    fmt = 115
    output_filename = f"申報資料解析結果{fmt}.csv"

    conn = get_conn()
    cursor = conn.cursor()
    sql = """SELECT MAX([End]) FROM [Hospital_data].[dbo].[CancerRegistry_Fields] WHERE fmt=? GROUP BY [fmt]"""
    cursor.execute(sql, fmt)
    rows = cursor.fetchone()
    expected_length = rows[0] if rows else 0
    conn.close()

    is_valid = True
    with open(file_name, "r", encoding="big5") as f:
        for i, line in enumerate(f):
            clean_line = line.strip('\n').strip('\r')
            if not clean_line.strip():
                continue
            
            length = len(clean_line.encode('big5'))
            if length != expected_length:
                print(f"第 {i+1} 行長度: {length} (預期: {expected_length})")
                is_valid = False

    if is_valid:
        print(f"資料符合{fmt}格式...")
        main(file_name, output_filename, fmt)
    else:
        print(f"資料未符合{fmt}格式")

資料符合115格式...
count:4row data
